# Config

> Configuration dataclasses and enums for the file browser.

In [ ]:
#| default_exp core.config

In [ ]:
#| export
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, List, Literal, Optional, Tuple

from cjm_file_discovery.core.models import FileInfo

## Enums

Enumeration types for browser configuration options.

In [ ]:
#| export
class SelectionMode(str, Enum):
    """File selection modes."""
    NONE = "none"           # Browse only, no selection
    SINGLE = "single"       # Select one file
    MULTIPLE = "multiple"   # Select multiple files

In [ ]:
#| export
class FileColumn(str, Enum):
    """Available columns for list view."""
    NAME = "name"
    SIZE = "size"
    MODIFIED = "modified"
    TYPE = "type"
    EXTENSION = "extension"
    PATH = "path"

In [ ]:
# Test enums
assert SelectionMode.SINGLE.value == "single"
assert FileColumn.NAME.value == "name"
print("Enum tests passed!")

## ViewConfig

Configuration for how directory contents are displayed.

In [ ]:
#| export
@dataclass
class ViewConfig:
    """View display configuration."""
    # List view columns
    columns: List[FileColumn] = field(default_factory=lambda: [
        FileColumn.NAME, FileColumn.SIZE, FileColumn.MODIFIED
    ])
    
    # Sorting
    default_sort: FileColumn = FileColumn.NAME   # Default sort column
    sort_folders_first: bool = True              # Sort folders before files
    allow_sort_toggle: bool = True               # Allow user to change sort

In [ ]:
# Test ViewConfig defaults
view_config = ViewConfig()
assert FileColumn.NAME in view_config.columns
assert view_config.sort_folders_first == True
assert view_config.allow_sort_toggle == True

# Test custom config
custom_view = ViewConfig(
    columns=[FileColumn.NAME, FileColumn.TYPE],
)
assert len(custom_view.columns) == 2

print("ViewConfig tests passed!")

## FilterConfig

Configuration for filtering which files are displayed.

In [ ]:
#| export
@dataclass
class FilterConfig:
    """Filtering configuration."""
    allowed_extensions: Optional[List[str]] = None   # Include only these extensions (None = all)
    show_directories: bool = True                    # Show directories
    show_hidden: bool = False                        # Show hidden files/directories
    show_system_files: bool = False                  # Show system files
    custom_filter: Optional[Callable[[FileInfo], bool]] = None  # Custom filter function
    
    def matches(
        self,
        file_info: FileInfo  # File to check against filter
    ) -> bool:  # True if file passes filter
        """Check if a file matches the filter criteria."""
        # Check directory visibility
        if file_info.is_directory and not self.show_directories:
            return False
        
        # Check hidden files
        if not self.show_hidden and file_info.name.startswith('.'):
            return False
        
        # Check extension filter (only for files)
        if not file_info.is_directory and self.allowed_extensions is not None:
            ext = file_info.extension or ''
            # Normalize extensions (handle with/without dot)
            allowed = [e.lstrip('.').lower() for e in self.allowed_extensions]
            if ext.lower() not in allowed:
                return False
        
        # Check custom filter
        if self.custom_filter is not None:
            return self.custom_filter(file_info)
        
        return True

In [ ]:
# Test FilterConfig
filter_config = FilterConfig()

# Test default (no filter)
file1 = FileInfo(name="test.txt", path="/test.txt", is_directory=False, extension="txt")
assert filter_config.matches(file1) == True

# Test hidden file filtering
hidden_file = FileInfo(name=".hidden", path="/.hidden", is_directory=False)
assert filter_config.matches(hidden_file) == False  # Hidden files off by default

filter_with_hidden = FilterConfig(show_hidden=True)
assert filter_with_hidden.matches(hidden_file) == True

# Test extension filtering
ext_filter = FilterConfig(allowed_extensions=[".py", ".txt"])
py_file = FileInfo(name="script.py", path="/script.py", is_directory=False, extension="py")
js_file = FileInfo(name="script.js", path="/script.js", is_directory=False, extension="js")
folder = FileInfo(name="folder", path="/folder", is_directory=True)

assert ext_filter.matches(py_file) == True
assert ext_filter.matches(js_file) == False
assert ext_filter.matches(folder) == True  # Directories always pass extension filter

# Test directory filtering
no_dirs = FilterConfig(show_directories=False)
assert no_dirs.matches(folder) == False
assert no_dirs.matches(py_file) == True

# Test custom filter
def size_filter(f: FileInfo) -> bool:
    return f.size is None or f.size < 1000

custom_filter = FilterConfig(custom_filter=size_filter)
small_file = FileInfo(name="small.txt", path="/small.txt", is_directory=False, size=100)
large_file = FileInfo(name="large.txt", path="/large.txt", is_directory=False, size=10000)
assert custom_filter.matches(small_file) == True
assert custom_filter.matches(large_file) == False

print("FilterConfig tests passed!")

FilterConfig tests passed!


## FileBrowserCallbacks

Optional callbacks for customizing browser behavior.

In [ ]:
#| export
@dataclass
class FileBrowserCallbacks:
    """Event callbacks for customization."""
    on_navigate: Optional[Callable[[str], None]] = None          # Called when directory changes
    on_select: Optional[Callable[[str], None]] = None            # Called when file selected
    on_selection_change: Optional[Callable[[List[str]], None]] = None  # Called when selection changes
    on_open: Optional[Callable[[str], None]] = None              # Called on file double-click/enter
    validate_selection: Optional[Callable[[str], Tuple[bool, str]]] = None  # Validate before select
    validate_navigation: Optional[Callable[[str], Tuple[bool, str]]] = None  # Validate before navigate

In [ ]:
# Test FileBrowserCallbacks
navigated_paths = []
selected_files = []

def track_navigate(path: str):
    navigated_paths.append(path)

def track_select(path: str):
    selected_files.append(path)

def validate_py_only(path: str) -> Tuple[bool, str]:
    if path.endswith('.py'):
        return (True, "")
    return (False, "Only .py files allowed")

callbacks = FileBrowserCallbacks(
    on_navigate=track_navigate,
    on_select=track_select,
    validate_selection=validate_py_only
)

# Test callbacks work
callbacks.on_navigate("/home/user")
callbacks.on_select("/home/user/file.txt")
assert navigated_paths == ["/home/user"]
assert selected_files == ["/home/user/file.txt"]

# Test validation
valid, msg = callbacks.validate_selection("/test.py")
assert valid == True
valid, msg = callbacks.validate_selection("/test.txt")
assert valid == False
assert "Only .py files" in msg

print("FileBrowserCallbacks tests passed!")

FileBrowserCallbacks tests passed!


## FileBrowserConfig

Main configuration for the file browser component.

In [ ]:
#| export
@dataclass
class FileBrowserConfig:
    """Main configuration for file browser."""
    # Provider
    provider: Optional[Any] = None     # FileSystemProvider, defaults to local
    start_path: Optional[str] = None   # Initial path (defaults to home)
    
    # Selection
    selection_mode: SelectionMode = SelectionMode.SINGLE
    selectable_types: Literal["files", "directories", "both"] = "files"
    max_selections: Optional[int] = None  # For MULTIPLE mode
    
    # Display
    view: ViewConfig = field(default_factory=ViewConfig)
    filter: FilterConfig = field(default_factory=FilterConfig)
    
    # Path bar
    show_path_bar: bool = True
    show_path_input: bool = False      # Direct path entry field
    show_breadcrumbs: bool = True
    
    # Bookmarks
    bookmarks: Optional[List[Tuple[str, str]]] = None  # [(label, path), ...]
    show_bookmarks: bool = False
    
    # HTML IDs (for HTMX targeting)
    container_id: str = "file-browser"
    content_id: str = "file-browser-content"
    path_bar_id: str = "file-browser-path"
    listing_id: str = "file-browser-listing"
    
    # Virtual collection
    vc_prefix: str = ""                # Prefix for virtual collection IDs (auto-generated if empty)
    
    def can_select(
        self,
        file_info: FileInfo  # File to check
    ) -> bool:  # True if file can be selected
        """Check if a file/directory can be selected based on config."""
        if self.selection_mode == SelectionMode.NONE:
            return False
        
        if self.selectable_types == "files" and file_info.is_directory:
            return False
        if self.selectable_types == "directories" and not file_info.is_directory:
            return False
        
        return True

In [ ]:
# Test FileBrowserConfig
config = FileBrowserConfig()

# Test defaults
assert config.selection_mode == SelectionMode.SINGLE
assert config.selectable_types == "files"
assert config.show_path_bar == True
assert config.container_id == "file-browser"
assert config.vc_prefix == ""
assert isinstance(config.view, ViewConfig)
assert isinstance(config.filter, FilterConfig)

# Test can_select
file = FileInfo(name="test.txt", path="/test.txt", is_directory=False)
folder = FileInfo(name="folder", path="/folder", is_directory=True)

# Default: files only
assert config.can_select(file) == True
assert config.can_select(folder) == False

# Directories only
dir_config = FileBrowserConfig(selectable_types="directories")
assert dir_config.can_select(file) == False
assert dir_config.can_select(folder) == True

# Both
both_config = FileBrowserConfig(selectable_types="both")
assert both_config.can_select(file) == True
assert both_config.can_select(folder) == True

# No selection
no_select = FileBrowserConfig(selection_mode=SelectionMode.NONE)
assert no_select.can_select(file) == False
assert no_select.can_select(folder) == False

# Test with custom IDs and vc_prefix
custom_config = FileBrowserConfig(
    container_id="my-browser",
    content_id="my-content",
    filter=FilterConfig(allowed_extensions=[".db"]),
    bookmarks=[("Home", "/home"), ("Downloads", "/downloads")],
    vc_prefix="fb",
)
assert custom_config.container_id == "my-browser"
assert custom_config.filter.allowed_extensions == [".db"]
assert len(custom_config.bookmarks) == 2
assert custom_config.vc_prefix == "fb"

print("FileBrowserConfig tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()